In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/competitions/earthquake-intensity-prediction-challenge/Train.csv
/kaggle/input/competitions/earthquake-intensity-prediction-challenge/Test.csv


In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

data_path = "/kaggle/input/competitions/earthquake-intensity-prediction-challenge"

train = pd.read_csv(os.path.join(data_path, "Train.csv"))
test = pd.read_csv(os.path.join(data_path, "Test.csv"))

target_col = "Magnitude"

X = train.drop(columns=[target_col])
y = train[target_col]
X_test = test.copy()

# Simple cleaning: impute missing with train means
train_means = X.mean()
X_filled = X.fillna(train_means)
X_test_filled = X_test.fillna(train_means)

# Optional: remove constant columns
nunique = X_filled.nunique()
constant_cols = nunique[nunique <= 1].index.tolist()
X_clean = X_filled.drop(columns=constant_cols)
X_test_clean = X_test_filled.drop(columns=constant_cols)

# Train/validation split
X_tr, X_val, y_tr, y_val = train_test_split(
    X_clean, y, test_size=0.2, random_state=42
)

model = RandomForestRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)
model.fit(X_tr, y_tr)

y_pred_val = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
print("Validation RMSE:", rmse)

# Train on full cleaned data and predict test
model.fit(X_clean, y)
test_preds = model.predict(X_test_clean)

Validation RMSE: 0.759289146846258


In [3]:
# IDs: 1, 2, ..., N
n_test = len(test)
ids = np.arange(1, n_test + 1)

submission = pd.DataFrame({
    "ID": ids,
    "TARGET": test_preds
})

submission.to_csv("Solution.csv", index=False)
print(submission.head())

   ID    TARGET
0   1  5.185667
1   2  5.752000
2   3  6.117333
3   4  4.459000
4   5  5.016667
